# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsf-rawnak/FlyRankAI-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Scoring / Ranking, built on top of a binary classification target.**

Lane 2's real question — "which pages should a reviewer look at first?" — is a ranking question,
not a plain yes/no question. Nobody just wants a list of pages flagged "problem" with no order;
they want the top of a queue that matches their actual weekly capacity (20-50 pages, not 13,000).

The way I'll get there is the standard two-step pattern from the lane guide: train a binary
classifier that outputs a *probability* a page is declining (a classification component), then
use that probability as the sort key for a ranked queue (the scoring/ranking output). So under
the hood it's classification; the deliverable a human actually uses is a rank-ordered list.


## 2. Target or proxy

**Target for this notebook: `is_declining_label = (trend_direction == "down")`** — the same
proxy the starter pipeline uses. I'm naming it a **proxy, not an observed outcome**, because
`trend_direction` is a bucket computed from the CURRENT 90-day window (`trend_pct`), not something
that happened *after* a decision point. A model trained on it can only learn "which pages look
like they're currently declining," not "which pages will decline next."

**Where a stronger label would come from:** the warehouse's `fact_content_daily_performance`
table, built as `features from a prior window -> label from a later window` (e.g. prior 90 days
of signals predicting decline over the next 30 days). That's the label I'm working toward for
ML-04/05 — this week I'm using the starter proxy because it's what's available in-repo right now,
and because the framing exercise (task type, metric, unit of analysis) is the same regardless of
which label backs it.


In [2]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Base rate of the proxy label: {df['is_declining_label'].mean():.1%} of pages currently flagged 'down'")
print(f"Rows: {len(df):,} | Positive (declining): {df['is_declining_label'].sum():,} | Negative: {(1 - df['is_declining_label']).sum():,}")


Base rate of the proxy label: 54.2% of pages currently flagged 'down'
Rows: 30,000 | Positive (declining): 16,262 | Negative: 13,738


## 3. Success metric

**Primary metric: Precision@50.** A reviewer works through the top of the queue, not the whole
list — so the number that actually matters is "of the top 50 pages the system says to check
first, how many are genuinely worth checking?" That maps directly to real reviewer capacity,
unlike accuracy or a raw AUC number, which score the whole 30,000-row list including the parts
nobody will ever look at.

**Secondary/supporting metrics:** ROC-AUC and average precision, to see whether the model's
ranking ability holds up across the whole list, not just the top 50 — useful for catching a model
that's good at the top but falls apart quickly after.

**What "good" means here, concretely:** the starter baseline rule already gets 24% of its top 50
picks right (0.240 precision@50). Beating that meaningfully — the starter's random forest reaches
74% — is the bar. If I can't clear the fixed-rule baseline with a real model, that's a real
finding too, not a failure to hide.


## 4. The unit of analysis, as a real dataframe

**One row = one content item (`content_id`), snapshotted with its trailing-90-day search and
engagement metrics.** That's the grain of the starter CSV, and it's the grain I'm scoring at —
one score per page, not per client, not per day.


In [3]:
lane2_cols = [
    "content_id", "client_id", "content_type",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "avg_position", "ctr", "engagement_rate",
    "word_count", "days_since_last_update", "content_age_days",
    "trend_direction", "trend_pct", "is_declining_label",
]

lane2_slice = df[lane2_cols]
print(f"Unit of analysis: one row = one content item. Shape: {lane2_slice.shape}")
lane2_slice.head()


Unit of analysis: one row = one content item. Shape: (30000, 15)


,content_id,client_id,content_type,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,word_count,days_since_last_update,content_age_days,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,17,10.6,0.76,5.88,3221.0,20,187,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,9,20.3,0.05,0.00,2481.0,25,445,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,11,36.5,0.09,0.00,3515.0,20,141,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,78,6.2,0.49,1.28,NaN,22,463,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,145,44.0,0.13,0.00,2803.0,14,263,down,-34.7,1


## 5. Why ML beats a fixed rule here

The starter baseline is a fixed-weight rule: `0.40*visibility + 0.30*freshness + 0.25*position +
0.05*depth_gap`. That works okay, but a fixed rule assumes each signal matters the same amount
for every page, which isn't true in this data.

**Real evidence:** I checked how strongly each individual signal correlates with the decline
label on its own. If one or two signals cleanly separated decliners from non-decliners, a simple
if-statement rule would be enough — no model needed. That's not what happens:


In [4]:
cols = ["days_since_last_update", "avg_position", "word_count", "engagement_rate", "ctr", "impressions_90d"]
corrs = df[cols + ["is_declining_label"]].corr()["is_declining_label"].drop("is_declining_label")
print("Correlation of each individual signal with the decline label:")
print(corrs.sort_values(key=abs, ascending=False).round(3))


Correlation of each individual signal with the decline label:
word_count                0.090
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
impressions_90d          -0.018
engagement_rate          -0.013
Name: is_declining_label, dtype: float64


No single signal clears |0.09| correlation with the label — `word_count` (0.090) and
`days_since_last_update` (0.081) are the strongest, and even those are weak alone. That's the
"messy but real" pattern the framing skill describes: the true signal is probably some
combination of several weak, tangled features rather than one dominant one — which is exactly the
kind of pattern a learned model (logistic regression, tree, random forest) can pick up on and a
single hand-written if-statement can't. The starter results back this up: the model that
considers all features together (random forest, 0.750 ROC-AUC) beats the fixed-weight rule
(0.627 ROC-AUC) by a wide margin.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.